# Prepare YOLO Dataset
Input:
- Directory containing HLW camera trap MP4 videos
- COCO annotations with frame-video mapping

Output:
- Stratified Train/val/test dataset in YOLO format

In [1]:
import os
import sys
import json

from pathlib import Path
from sklearn.model_selection import train_test_split

from video_dataset_prep_tools import extract_single_video, compute_split_statistics, stratified_video_split, split_report, extract_frames_by_split
from annotation_converter import AnnotationConverter


In [2]:
video_dir = Path("../data/h23/Videos")  # Path containing raw camera trap videos
data_dir = Path("../data/h23")  # Path where YOLO dataset will be created

video_arr = [f for f in os.listdir(video_dir) if f.lower().endswith('.mp4')]

## Compute Video Statistics

In [3]:
video_stats_df = compute_split_statistics(coco_json_path="../data/h23/instances_merged.json")
display(video_stats_df)

,video,n_total_frames,n_bird_frames,prevalence,stratum
0,IMG_0019,303,120,0.396040,2
1,IMG_0032,303,303,1.000000,3
2,IMG_0042,303,1,0.003300,0
3,IMG_0049,303,3,0.009901,0
4,IMG_0050,303,1,0.003300,0
...,...,...,...,...,...
87,IMG_0255,303,87,0.287129,1
88,IMG_0256,303,102,0.336634,1
89,IMG_0257,303,76,0.250825,1
90,IMG_0258,303,82,0.270627,1


## Split Dataset

In [4]:
train_videos, val_videos, test_videos = stratified_video_split(video_stats_df, save_dir=data_dir)

split_report(video_stats_df, train_videos, "Train")
split_report(video_stats_df, val_videos,   "Val")
split_report(video_stats_df, test_videos,  "Test")

Split saved to ../data/h23/split.json
Train  |  64 videos |  19381 frames | bird prevalence: 0.410 ± 0.379
Val    |  16 videos |   4843 frames | bird prevalence: 0.393 ± 0.402
Test   |  12 videos |   3636 frames | bird prevalence: 0.404 ± 0.379


## Extract Frames into YOLO Image Directories

In [ ]:
out_dir = data_dir / "images"
split_json = data_dir / "split.json"
    
extract_frames_by_split(split_json, video_dir, out_dir)

KeyboardInterrupt: 

## Convert COCO annotations to YOLO format

In [8]:
def coco_to_yolo_split(train_videos, val_videos, test_videos):
    converter = AnnotationConverter(class_mapping={'bird': 0})
    coco_json_path = "../data/h23/instances_merged.json"

    splits = {
        "train": train_videos,
        "val":   val_videos,
        "test":  test_videos,
    }

    for split_name, videos in splits.items():
        output_dir = data_dir / "labels" / split_name
        converter.coco_to_yolo(
            coco_json_path=coco_json_path,
            output_dir=output_dir,
            use_filename=True,
            video_filter=videos,
        )
        print(f"[{split_name:5s}] labels → {output_dir}")

    converter.create_yaml_config(
    output_dir=data_dir,
    dataset_path='data/h23',
    train_dir='images/train',
    val_dir='images/val',
    test_dir='images/test'
)

coco_to_yolo_split(train_videos, val_videos, test_videos)

Converted 7944 images from COCO to YOLO format
Output directory: ../data/h23/labels/train
[train] labels → ../data/h23/labels/train
Converted 1904 images from COCO to YOLO format
Output directory: ../data/h23/labels/val
[val  ] labels → ../data/h23/labels/val
Converted 1469 images from COCO to YOLO format
Output directory: ../data/h23/labels/test
[test ] labels → ../data/h23/labels/test
Created YOLO config: ../data/h23/yolo.yaml
Classes: {'bird': 0, 'Bird': 0}
